# 图像分类工程：从数据合同到可拒识的批量推理

这份 notebook 用 scikit-learn 内置的 8×8 手写数字和受控扰动，走通一条可离线复现的图像分类链路。重点不是追求榜单精度，而是回答：**图像 shape、色彩空间和数值范围怎样成为接口；如何阻止同源图片泄漏；怎样评估每类错误、置信度、模糊/噪声与 OOD；模型和预处理如何一起发布？**

全程不下载数据。这里的线性分类器是工程基线，不能代表现代 CNN/ViT 在真实图片上的最终效果。

## 1. 先画清请求链路和验收面

```text
原始样本 + source_id + captured_at
  -> 数据合同校验（shape/dtype/color/range）
  -> 按 source/group 或时间切分
  -> 仅训练集增强
  -> 版本化 resize/normalize
  -> baseline fit + calibration audit
  -> class / confidence / reject_reason
  -> 分群指标、漂移监控与可回滚发布
```

离线高 accuracy 只覆盖其中一小段。工程验收还要检查：训练与测试是否共享同一来源、服务是否接受错误通道顺序、低置信和分布外输入如何处理、批量输出能否追溯到模型与预处理版本。

In [ ]:
from dataclasses import dataclass
from hashlib import sha256
import json
import math
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix,
                             f1_score, log_loss, precision_recall_fscore_support)

SEED = 23
rng = np.random.default_rng(SEED)
digits = load_digits()
base_images = digits.images.astype(np.float32)  # 官方数据合同：灰度值 0..16
base_labels = digits.target.astype(np.int64)
print({"samples": len(base_images), "shape": base_images.shape,
       "classes": np.unique(base_labels).tolist(),
       "range": (float(base_images.min()), float(base_images.max()))})


## 2. 图像不是一个无类型的 ndarray

一个可发布的数据合同至少写明：批维、空间尺寸、通道顺序、颜色空间、dtype、闭区间范围、缺失/非有限值策略和 resize 规则。`[N,H,W]` 灰度图与 `[N,H,W,C]` RGB 即使元素数相同也不是同一接口；把 0..255 当作 0..1 会造成静默分布偏移。

下面的预处理器固定为 `8×8 GRAY 0..16 -> 16×16 float32 0..1`。最近邻 resize 是为了把坐标与合同讲清楚，不是生产图像质量建议。真实系统还要锁定插值算法、EXIF 旋转、alpha 合成和 ICC profile。

In [ ]:
@dataclass(frozen=True)
class ImageContract:
    height: int = 8
    width: int = 8
    color_space: str = "GRAY"
    min_value: float = 0.0
    max_value: float = 16.0
    preprocess_version: str = "digits-gray-nn16-v1"

    def validate(self, batch: np.ndarray) -> None:
        if not isinstance(batch, np.ndarray) or batch.ndim != 3:
            raise ValueError("期望 [N,H,W] numpy batch")
        if batch.shape[1:] != (self.height, self.width):
            raise ValueError(f"空间尺寸错误: {batch.shape[1:]}")
        if not np.issubdtype(batch.dtype, np.number) or not np.isfinite(batch).all():
            raise ValueError("像素必须是有限数值")
        if batch.size and (batch.min() < self.min_value or batch.max() > self.max_value):
            raise ValueError("像素范围违反 0..16 合同")

    def transform(self, batch: np.ndarray) -> np.ndarray:
        self.validate(batch)
        # 显式最近邻 2x；算法改变就必须升级 preprocess_version。
        resized = np.repeat(np.repeat(batch, 2, axis=1), 2, axis=2)
        return (resized / self.max_value).astype(np.float32)

contract = ImageContract()
contract.validate(base_images)
transformed = contract.transform(base_images[:2])
assert transformed.shape == (2, 16, 16)
assert transformed.dtype == np.float32
assert 0.0 <= transformed.min() <= transformed.max() <= 1.0
print(contract, transformed.shape)


## 3. 切分单位应是“来源”，不一定是图片行

截图连拍、同一病人的多张切片、同一商品的不同角度，往往共享强烈的背景或主体特征。若先生成增强副本再随机按行切分，同一个 source 的近重复图会同时进入训练和测试，指标会虚高。

我们为每张原图制造一个微扰副本，并让二者共享 `source_id`。随后对比按行随机切分和 `GroupShuffleSplit`。若业务预测未来批次，还应按 `captured_at` 做时间外推测试；不能看见未来分布后再随机打散。

In [ ]:
noise = rng.normal(0, 0.20, size=base_images.shape).astype(np.float32)
variant_images = np.concatenate([base_images, np.clip(base_images + noise, 0, 16)])
variant_labels = np.concatenate([base_labels, base_labels])
source_ids = np.concatenate([np.arange(len(base_images)), np.arange(len(base_images))])
captured_at = np.concatenate([np.arange(len(base_images)), np.arange(len(base_images))])

row_train, row_test = train_test_split(
    np.arange(len(variant_images)), test_size=0.2, random_state=SEED,
    stratify=variant_labels
)
leaked_sources = set(source_ids[row_train]) & set(source_ids[row_test])

outer = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_val_idx, test_idx = next(outer.split(variant_images, variant_labels, source_ids))
inner = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED + 1)
train_rel, val_rel = next(inner.split(variant_images[train_val_idx],
                                      variant_labels[train_val_idx],
                                      source_ids[train_val_idx]))
train_idx = train_val_idx[train_rel]
val_idx = train_val_idx[val_rel]

assert leaked_sources, "按行随机切分理应展示近重复泄漏"
assert not (set(source_ids[train_idx]) & set(source_ids[val_idx]))
assert not (set(source_ids[train_idx]) & set(source_ids[test_idx]))
assert not (set(source_ids[val_idx]) & set(source_ids[test_idx]))

# 时间切分只展示合同：较早 source 训练，较晚 source 测试。
time_cut = int(np.quantile(np.unique(captured_at), 0.8))
time_train = np.flatnonzero(captured_at < time_cut)
time_test = np.flatnonzero(captured_at >= time_cut)
assert captured_at[time_train].max() < captured_at[time_test].min()
print("row split 泄漏 source 数:", len(leaked_sources),
      "group split rows:", len(train_idx), len(val_idx), len(test_idx))


## 4. 增强是训练期随机过程

增强必须在切分之后，只从训练样本产生；validation/test 保持确定性，才能比较版本。增强策略还要符合标签不变性：数字轻微平移通常不改标签，但 6/9 的任意旋转就可能改变语义。医疗、遥感等场景的左右翻转也未必安全。

下面只做一像素右移并清零边界，同时记录增强版本。生产 pipeline 应保存随机种子或样本级参数，便于重放困难样本。

In [ ]:
AUGMENT_VERSION = "shift-right-1-v1"

def shift_right(batch: np.ndarray) -> np.ndarray:
    shifted = np.zeros_like(batch)
    shifted[:, :, 1:] = batch[:, :, :-1]
    return shifted

train_raw = variant_images[train_idx]
train_labels = variant_labels[train_idx]
train_aug = np.concatenate([train_raw, shift_right(train_raw)])
y_train_aug = np.concatenate([train_labels, train_labels])

X_train = contract.transform(train_aug).reshape(len(train_aug), -1)
X_val = contract.transform(variant_images[val_idx]).reshape(len(val_idx), -1)
X_test = contract.transform(variant_images[test_idx]).reshape(len(test_idx), -1)
y_val, y_test = variant_labels[val_idx], variant_labels[test_idx]

assert len(X_train) == 2 * len(train_idx)
assert len(X_val) == len(val_idx) and len(X_test) == len(test_idx)
print("train/val/test feature shapes:", X_train.shape, X_val.shape, X_test.shape)


## 5. 先用可解释基线验证数据链路

线性 softmax 分类器能快速暴露标签错位、预处理不一致和切分错误，也提供后续 CNN/ViT 的最低基线。训练只读取训练集；阈值选择只读取 validation；test 最后一次性报告。

这里的 8×8 灰度数字与真实高分辨率、多域、多摄像头数据差距很大。不要把本结果写成“模型达到生产精度”，也不要根据 test 反复调参。

In [ ]:
classifier = LogisticRegression(
    C=2.0, max_iter=900, solver="lbfgs", random_state=SEED
)
classifier.fit(X_train, y_train_aug)
val_prob = classifier.predict_proba(X_val)
test_prob = classifier.predict_proba(X_test)
val_pred = classifier.classes_[val_prob.argmax(axis=1)]
test_pred = classifier.classes_[test_prob.argmax(axis=1)]
assert np.allclose(test_prob.sum(axis=1), 1.0)
print("validation accuracy:", round(accuracy_score(y_val, val_pred), 4),
      "test accuracy:", round(accuracy_score(y_test, test_pred), 4))


## 6. 总准确率会隐藏小类和业务代价

至少同时看 confusion matrix、每类 precision/recall/F1、macro-F1、balanced accuracy 和样本分群指标。micro-F1 在单标签多类任务中通常接近 accuracy，不能代替小类分析。若不同错误代价不同，还需业务 cost matrix；例如“良性判成恶性”和反向错误不应等价。

In [ ]:
cm = confusion_matrix(y_test, test_pred, labels=classifier.classes_)
precision, recall, per_class_f1, support = precision_recall_fscore_support(
    y_test, test_pred, labels=classifier.classes_, zero_division=0
)
metrics = {
    "accuracy": accuracy_score(y_test, test_pred),
    "balanced_accuracy": balanced_accuracy_score(y_test, test_pred),
    "macro_f1": f1_score(y_test, test_pred, average="macro"),
    "micro_f1": f1_score(y_test, test_pred, average="micro"),
    "log_loss": log_loss(y_test, test_prob, labels=classifier.classes_),
}
worst = np.argsort(per_class_f1)[:3]
print({k: round(float(v), 4) for k, v in metrics.items()})
print("最低 F1 类别:", [(int(classifier.classes_[i]), round(float(per_class_f1[i]), 3), int(support[i])) for i in worst])
assert cm.sum() == len(y_test)
assert len(per_class_f1) == 10


## 7. 错误分析要回到可追溯样本

保存 `sample_id/source_id、真值、预测、概率、模型版本、预处理版本`，再按混淆对、来源设备、时间、清晰度和人工标签质量聚合。仅展示“错了几张”无法判断该修数据、标签、模型还是输入合同。

margin 是最高与次高概率之差；小 margin 常提示类别边界不确定，但它不是校准后的错误概率。

In [ ]:
sorted_prob = np.sort(test_prob, axis=1)
margins = sorted_prob[:, -1] - sorted_prob[:, -2]
wrong = np.flatnonzero(test_pred != y_test)
error_rows = [
    {
        "row": int(test_idx[i]), "source_id": int(source_ids[test_idx[i]]),
        "truth": int(y_test[i]), "pred": int(test_pred[i]),
        "confidence": round(float(test_prob[i].max()), 4),
        "margin": round(float(margins[i]), 4),
    }
    for i in wrong[:8]
]
print("errors:", len(wrong), "examples:", error_rows[:3])
if len(wrong):
    assert all(row["truth"] != row["pred"] for row in error_rows)


## 8. 置信度、概率校准与拒识是三个不同问题

softmax 最大值不是天然正确率。Expected Calibration Error（ECE）把置信度分箱，比较每箱平均置信与实际正确率；它依赖分箱且会掩盖局部问题，因此还应看 reliability table、NLL/Brier 及关键分群。**下面只审计未校准概率，并没有实现 temperature scaling、Platt 或 isotonic calibration**；若要拟合校准器，还要单独保留 calibration split，不能和选拒识阈值的数据反复复用。

拒识阈值应在 validation 上按覆盖率与已接收样本准确率选择，再冻结后评估 test。下面选择满足 validation 已接收准确率目标的最低阈值；无可行阈值时，会在 fallback 阈值上如实重算 coverage/accuracy，而不是伪造零覆盖率。真实系统应把人工复核容量和漏判代价写进目标函数。

In [ ]:
def calibration_table(y_true, probabilities, classes, bins=10):
    confidence = probabilities.max(axis=1)
    prediction = classes[probabilities.argmax(axis=1)]
    edges = np.linspace(0.0, 1.0, bins + 1)
    rows, ece = [], 0.0
    for index in range(bins):
        mask = (confidence >= edges[index]) & (confidence < edges[index + 1] if index < bins - 1 else confidence <= edges[index + 1])
        if not mask.any():
            continue
        accuracy = np.mean(prediction[mask] == y_true[mask])
        mean_conf = confidence[mask].mean()
        ece += mask.mean() * abs(accuracy - mean_conf)
        rows.append((index, int(mask.sum()), float(mean_conf), float(accuracy)))
    return float(ece), rows

def select_reject_threshold(y_true, probabilities, classes, target_accuracy=0.97,
                            fallback_threshold=0.98):
    confidence = probabilities.max(axis=1)
    prediction = classes[probabilities.argmax(axis=1)]
    feasible = []
    for threshold in np.linspace(0.40, 0.98, 30):
        accepted = confidence >= threshold
        if accepted.any():
            acc = np.mean(prediction[accepted] == y_true[accepted])
            if acc >= target_accuracy:
                feasible.append((float(threshold), float(accepted.mean()), float(acc), True))
    if feasible:
        return max(feasible, key=lambda row: (row[1], -row[0]))
    accepted = confidence >= fallback_threshold
    coverage = float(accepted.mean())
    accuracy = float(np.mean(prediction[accepted] == y_true[accepted])) if accepted.any() else math.nan
    return float(fallback_threshold), coverage, accuracy, False

# ECE 是未校准概率的审计量，不是校准算法。
ece, reliability = calibration_table(y_val, val_prob, classifier.classes_)
reject_threshold, val_coverage, val_accepted_acc, id_threshold_feasible = select_reject_threshold(
    y_val, val_prob, classifier.classes_, target_accuracy=0.97
)
test_conf = test_prob.max(axis=1)
test_accepted = test_conf >= reject_threshold
test_accepted_acc = np.mean(test_pred[test_accepted] == y_test[test_accepted]) if test_accepted.any() else math.nan
print({"audit": "uncalibrated_probability", "val_ece": round(ece, 4),
       "id_threshold": round(reject_threshold, 3), "id_threshold_feasible": id_threshold_feasible,
       "val_coverage": round(val_coverage, 3),
       "test_coverage_at_id_threshold": round(float(test_accepted.mean()), 3),
       "test_accepted_accuracy": round(float(test_accepted_acc), 3)})
assert 0.0 <= ece <= 1.0
assert 0.0 <= reject_threshold <= 1.0


## 9. 模糊、噪声与 OOD 要单独建验证集和测试集

普通 test 与训练分布相似，无法证明摄像头失焦、压缩噪声、遮挡或未知类别下可靠。我们用均值模糊、固定噪声和均匀随机图构造**受控压力测试**：只用 OOD validation 冻结最大 softmax gate，再在独立 OOD test 报 acceptance；同时手算 AUROC 与 FPR95。test OOD 不能反向改阈值。

这仍不是完整 OOD 检测器：均匀噪声通常比真实近分布未知类容易，最大 softmax 的 AUROC/FPR95 和 ID coverage 也会暴露明显代价。生产应收集真实失败分布，比较 energy/embedding/dedicated detector，监控输入统计和置信度漂移，并对未知类、空白图、超大图及解码炸弹设置入口保护。

In [ ]:
def mean_blur(batch: np.ndarray) -> np.ndarray:
    padded = np.pad(batch, ((0, 0), (1, 1), (1, 1)), mode="edge")
    total = np.zeros_like(batch, dtype=np.float32)
    for dy in range(3):
        for dx in range(3):
            total += padded[:, dy:dy + batch.shape[1], dx:dx + batch.shape[2]]
    return total / 9.0

def evaluate_corruption(raw_batch, labels):
    features = contract.transform(np.clip(raw_batch, 0, 16)).reshape(len(raw_batch), -1)
    probability = classifier.predict_proba(features)
    prediction = classifier.classes_[probability.argmax(axis=1)]
    return float(accuracy_score(labels, prediction)), float(probability.max(axis=1).mean())

def select_ood_confidence_gate(id_confidence, ood_confidence,
                               max_ood_acceptance=.10, min_id_coverage=.10):
    id_confidence = np.asarray(id_confidence, dtype=float)
    ood_confidence = np.asarray(ood_confidence, dtype=float)
    candidates = np.unique(np.r_[0.0, id_confidence, ood_confidence, 1.0])
    feasible = []
    for threshold in candidates:
        id_coverage = float(np.mean(id_confidence >= threshold))
        ood_acceptance = float(np.mean(ood_confidence >= threshold))
        if ood_acceptance <= max_ood_acceptance and id_coverage >= min_id_coverage:
            feasible.append((float(threshold), id_coverage, ood_acceptance, True))
    if feasible:
        return max(feasible, key=lambda row: (row[1], -row[2], -row[0]))
    observed = [(float(t), float(np.mean(id_confidence >= t)),
                 float(np.mean(ood_confidence >= t)), False) for t in candidates]
    return min(observed, key=lambda row: (row[2], -row[1], row[0]))

def pairwise_auroc(positive_scores, negative_scores):
    positive_scores = np.asarray(positive_scores, dtype=float)
    negative_scores = np.asarray(negative_scores, dtype=float)
    comparisons = positive_scores[:, None] - negative_scores[None, :]
    return float(np.mean(comparisons > 0) + .5 * np.mean(comparisons == 0))

raw_test = variant_images[test_idx]
blur_acc, blur_conf = evaluate_corruption(mean_blur(raw_test), y_test)
noisy = np.clip(raw_test + rng.normal(0, 2.5, raw_test.shape), 0, 16).astype(np.float32)
noise_acc, noise_conf = evaluate_corruption(noisy, y_test)

# validation OOD 只用于冻结 gate；独立 test OOD 最后一次报告，不能反向调阈值。
ood_val_rng = np.random.default_rng(SEED + 1001)
ood_test_rng = np.random.default_rng(SEED + 2002)
ood_val = ood_val_rng.uniform(0, 16, size=(len(X_val), 8, 8)).astype(np.float32)
ood = ood_test_rng.uniform(0, 16, size=(256, 8, 8)).astype(np.float32)
ood_val_prob = classifier.predict_proba(contract.transform(ood_val).reshape(len(ood_val), -1))
ood_prob = classifier.predict_proba(contract.transform(ood).reshape(len(ood), -1))
val_id_conf, val_ood_conf = val_prob.max(axis=1), ood_val_prob.max(axis=1)
ood_gate_threshold, _, _, ood_gate_feasible = select_ood_confidence_gate(
    val_id_conf, val_ood_conf, max_ood_acceptance=.10, min_id_coverage=.10
)
id_only_threshold = reject_threshold
reject_threshold = max(id_only_threshold, ood_gate_threshold)
final_val_id_coverage = float(np.mean(val_id_conf >= reject_threshold))
final_val_ood_acceptance = float(np.mean(val_ood_conf >= reject_threshold))
test_ood_acceptance = float(np.mean(ood_prob.max(axis=1) >= reject_threshold))
ood_auroc = pairwise_auroc(val_id_conf, val_ood_conf)
id_tpr95_threshold = float(np.quantile(val_id_conf, .05, method="lower"))
ood_fpr95 = float(np.mean(val_ood_conf >= id_tpr95_threshold))

robustness = {"clean_acc": metrics["accuracy"], "blur_acc": blur_acc,
              "noise_acc": noise_acc, "ood_auroc_validation": ood_auroc,
              "ood_fpr95_validation": ood_fpr95, "frozen_threshold": reject_threshold,
              "validation_id_coverage": final_val_id_coverage,
              "validation_ood_acceptance": final_val_ood_acceptance,
              "test_ood_acceptance": test_ood_acceptance,
              "ood_gate_feasible": ood_gate_feasible}
print({k: round(float(v), 4) if isinstance(v, (float, np.floating)) else v
       for k, v in robustness.items()})
assert 0.0 <= blur_acc <= 1.0 and 0.0 <= noise_acc <= 1.0
assert ood_prob.shape == (256, 10)
assert 0.0 <= ood_auroc <= 1.0 and 0.0 <= ood_fpr95 <= 1.0
assert ood_gate_feasible
assert final_val_ood_acceptance <= .10 + 1 / len(val_ood_conf)
assert test_ood_acceptance < .30


## 10. 批量推理发布的是 bundle，不只是一份权重

服务输入应有最大 batch/尺寸限制，输出保持与输入顺序一致，并携带 `model_version、preprocess_version、threshold_version`。预处理和类别映射是模型的一部分；只替换权重文件会产生难以察觉的线上错配。

真实部署还需要线程/进程安全、warm-up、超时、动态 batching、CPU/GPU 数值回归、模型签名和灰度回滚。这里用一个小 bundle 固定外部合同。

In [ ]:
@dataclass
class ClassificationBundle:
    estimator: object
    contract: ImageContract
    reject_threshold: float
    model_version: str
    threshold_version: str
    max_batch: int = 128

    def predict(self, batch: np.ndarray):
        if len(batch) == 0 or len(batch) > self.max_batch:
            raise ValueError("batch 大小必须在 1..max_batch")
        features = self.contract.transform(batch).reshape(len(batch), -1)
        probabilities = self.estimator.predict_proba(features)
        positions = probabilities.argmax(axis=1)
        labels = self.estimator.classes_[positions]
        confidence = probabilities[np.arange(len(batch)), positions]
        return [
            {"label": int(label) if conf >= self.reject_threshold else None,
             "confidence": float(conf),
             "rejected": bool(conf < self.reject_threshold),
             "model_version": self.model_version,
             "preprocess_version": self.contract.preprocess_version,
             "threshold_version": self.threshold_version}
            for label, conf in zip(labels, confidence)
        ]

model_fingerprint = sha256(
    classifier.coef_.tobytes() + classifier.intercept_.tobytes()
).hexdigest()[:12]
bundle = ClassificationBundle(classifier, contract, reject_threshold,
                              f"logreg-{model_fingerprint}", "id97-oodval10-v2")
batch_result = bundle.predict(base_images[:5])
print(batch_result[:2])
assert len(batch_result) == 5
assert all(row["preprocess_version"] == contract.preprocess_version for row in batch_result)


## 11. 最小回归套件

回归测试应同时覆盖正常输入、边界输入、拒绝路径、确定性、类别映射和数据隔离。下面的断言是 notebook 可执行契约；生产 CI 还应固定黄金样本与容差，并分别在目标 CPU/GPU/推理引擎上运行。

In [ ]:
# 数据与预处理合同
assert base_images.shape[1:] == (8, 8)
assert base_labels.min() == 0 and base_labels.max() == 9
assert contract.transform(base_images[:1]).shape == (1, 16, 16)
assert np.isclose(contract.transform(np.zeros((1, 8, 8), dtype=np.float32)).sum(), 0.0)
try:
    contract.validate(np.zeros((1, 8, 8, 1), dtype=np.float32))
    raise AssertionError("错误 rank 应被拒绝")
except ValueError:
    pass
try:
    contract.validate(np.full((1, 8, 8), 17, dtype=np.float32))
    raise AssertionError("越界像素应被拒绝")
except ValueError:
    pass

# 切分、概率、拒识与 OOD 指标合同
assert set(source_ids[train_idx]).isdisjoint(source_ids[test_idx])
assert set(source_ids[val_idx]).isdisjoint(source_ids[test_idx])
assert classifier.classes_.tolist() == list(range(10))
assert test_prob.shape == (len(test_idx), 10)
assert np.all((test_prob >= 0) & (test_prob <= 1))
assert cm.shape == (10, 10) and cm.sum() == len(test_idx)
assert 0.0 <= metrics["macro_f1"] <= 1.0
assert len(reliability) >= 1
fallback_probe = np.array([[0.99, 0.01], [0.01, 0.99]])
probe_threshold, probe_coverage, probe_accuracy, probe_feasible = select_reject_threshold(
    np.array([0, 1]), fallback_probe, np.array([0, 1]), target_accuracy=1.1, fallback_threshold=0.98
)
assert not probe_feasible and probe_threshold == 0.98
assert probe_coverage == 1.0 and probe_accuracy == 1.0
assert bundle.reject_threshold == robustness["frozen_threshold"]
assert 0.0 <= robustness["ood_auroc_validation"] <= 1.0
assert robustness["validation_ood_acceptance"] <= .10 + 1 / len(val_ood_conf)
assert robustness["test_ood_acceptance"] < .30

# 服务合同与确定性
first = bundle.predict(base_images[:3])
second = bundle.predict(base_images[:3])
assert first == second
assert [r["model_version"] for r in first] == [bundle.model_version] * 3
assert bundle.threshold_version == "id97-oodval10-v2"
assert all((r["label"] is None) == r["rejected"] for r in first)
try:
    bundle.predict(base_images[:0])
    raise AssertionError("空 batch 应被拒绝")
except ValueError:
    pass
try:
    bundle.predict(np.repeat(base_images[:1], bundle.max_batch + 1, axis=0))
    raise AssertionError("超大 batch 应被拒绝")
except ValueError:
    pass
print("图像分类契约测试通过：数据、切分、指标、OOD、阈值和服务断言全部通过")


## 12. 失败边界、上线清单与延伸阅读

**常见失败**：按图片而非主体切分；先增强再切分；RGB/BGR 或 0..1/0..255 静默错配；按 test 调阈值；只报 accuracy；把最大 softmax 当 OOD 保证；发布权重却漏发类别表与预处理；离线 resize 和服务 resize 不同。

**上线前**：冻结 schema 与版本；按来源/时间做外推集；真实扰动与关键分群回归；阈值容量评估；空白/损坏/超大输入防护；延迟和峰值内存压测；shadow/canary；模型卡、数据卡、回滚与审计。

**资料**：

- scikit-learn `load_digits` 官方接口与数据范围：https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html
- Deng et al., *ImageNet: A Large-Scale Hierarchical Image Database*, CVPR 2009：https://www.image-net.org/static_files/papers/imagenet_cvpr09.pdf
- He et al., *Deep Residual Learning for Image Recognition*, CVPR 2016：https://arxiv.org/abs/1512.03385
- Guo et al., *On Calibration of Modern Neural Networks*, ICML 2017：https://proceedings.mlr.press/v70/guo17a.html

本 notebook 的受控数字数据、最近邻 resize 和线性模型只验证工程逻辑。现代生产分类通常需要经域内数据训练的 CNN/ViT、可靠校准和真实 OOD/漂移基准。